# Build Vector Index on Colab Pro GPU

Run this notebook on Colab Pro (T4 or better) to:
1. Clone your repo
2. Download FDA Orange Book
3. Build the ChromaDB index with bge-m3 on GPU (~30 sec vs ~25 min on CPU)
4. Save the ChromaDB folder to Google Drive
5. Download it to your local machine for fast querying

**Prerequisites:**
- Runtime → Change runtime type → T4 GPU
- Google Drive mounted

## 1. Mount Google Drive + clone repo

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Edit this path to where you want all project artifacts saved on Drive
DRIVE_ROOT = '/content/drive/MyDrive/pharma-intelligence'
DRIVE_OUTPUT = f'{DRIVE_ROOT}/chroma_db'

REPO_URL = 'https://github.com/siriponsri/pharma-intelligence.git'
REPO_DIR = '/content/pharma-intelligence'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists, pulling latest changes.')
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

Mounted at /content/drive
Cloning into '/content/pharma-intelligence'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 95 (delta 18), reused 90 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 122.18 KiB | 1.02 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/pharma-intelligence


## 2. Install dependencies

This notebook runs inside Google Colab's managed Jupyter environment, so do not install the repository's optional notebook extras here. Installing only the core project package avoids replacing Colab's pinned `ipykernel`, `jupyter-client`, and `notebook` packages.

If `pip` prints resolver warnings about packages preinstalled by Colab, you can continue as long as the install command completes successfully and the import check in the next cell passes.

In [2]:
# Install only the core project package in Colab.
# Do not install the optional notebook extra because Colab manages its own Jupyter stack.
%pip install -q -U pip setuptools wheel
%pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pharma-intel (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.41.0

In [3]:
import sys
from pathlib import Path

repo_dir = Path("/content/pharma-intelligence")
src_dir = repo_dir / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import pharma_intel
print("Python executable:", sys.executable)
print("pharma_intel imported from:", pharma_intel.__file__)

Python executable: /usr/bin/python3
pharma_intel imported from: /content/pharma-intelligence/src/pharma_intel/__init__.py


## 3. Set env vars (ThaiLLM API key)

⚠️ Do NOT commit the key. Use Colab's "Secrets" feature (🔑 icon in left sidebar) to store `THAILLM_API_KEY`, then load here:

In [4]:
import os
from google.colab import userdata

os.environ['THAILLM_API_KEY'] = userdata.get('THAILLM_API_KEY')
os.environ['LLM_PROVIDER'] = 'thaillm'

# Save project artifacts directly on Drive
os.environ['CHROMA_PERSIST_DIR'] = DRIVE_OUTPUT
os.environ['DATA_DIR'] = f'{DRIVE_ROOT}/data'
os.environ['MODEL_CACHE_DIR'] = f'{DRIVE_ROOT}/.cache/models'

for path in [os.environ['CHROMA_PERSIST_DIR'], os.environ['DATA_DIR'], os.environ['MODEL_CACHE_DIR']]:
    os.makedirs(path, exist_ok=True)

## 4. Verify GPU is available

In [5]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

CUDA available: True
Device: Tesla T4


## 5. Run the pipeline

The next cleanup cell makes the notebook rerun-safe by removing any stale ChromaDB folder from a previous failed or partial indexing run. It does not remove the model cache.

In [6]:
import os
import shutil
from pathlib import Path

persist_dir = Path(os.environ['CHROMA_PERSIST_DIR'])

if persist_dir.exists():
    shutil.rmtree(persist_dir)
    print(f'Removed existing ChromaDB directory: {persist_dir}')
persist_dir.mkdir(parents=True, exist_ok=True)
print(f'Ready for a fresh index build at: {persist_dir}')

Removed existing ChromaDB directory: /content/drive/MyDrive/pharma-intelligence/chroma_db
Ready for a fresh index build at: /content/drive/MyDrive/pharma-intelligence/chroma_db


In [7]:
from pharma_intel.ingestion.fda_orange_book import run_pipeline
from pharma_intel.rag.indexer import index_monographs

# Step 1: download + parse + filter
monographs = run_pipeline(force_download=False)
print(f'\n✓ {len(monographs)} monographs ready for indexing')

2026-04-22 03:42:50 | INFO     | pharma_intel.ingestion.fda_orange_book:download_orange_book:136 - Downloading Orange Book from https://www.fda.gov/media/76860/download...
2026-04-22 03:42:50 | INFO     | pharma_intel.ingestion.fda_orange_book:download_orange_book:140 - Downloaded 1.0 MB
2026-04-22 03:42:50 | INFO     | pharma_intel.ingestion.fda_orange_book:download_orange_book:144 - Extracted files: ['exclusivity.txt', 'patent.txt', 'products.txt']
2026-04-22 03:42:50 | INFO     | pharma_intel.ingestion.fda_orange_book:load_products:187 - Loaded 48083 product records from products.txt
2026-04-22 03:42:50 | INFO     | pharma_intel.ingestion.fda_orange_book:load_patents:194 - Loaded 20858 patent records from patent.txt
2026-04-22 03:42:50 | INFO     | pharma_intel.ingestion.fda_orange_book:load_exclusivity:201 - Loaded 2054 exclusivity records from exclusivity.txt
2026-04-22 03:42:51 | INFO     | pharma_intel.ingestion.fda_orange_book:filter_cardiometabolic:228 - Filtered to 5479 cardi


✓ 5479 monographs ready for indexing


In [8]:
# Step 2: embed with bge-m3 on GPU — this is the speedup
import time
start = time.time()
store = index_monographs(monographs, batch_size=64)   # larger batch on GPU
print(f'\n✓ Indexing took {time.time() - start:.1f} seconds')
print(f'✓ Collection size: {store.collection.count()}')

2026-04-22 03:42:58 | INFO     | pharma_intel.rag.indexer:index_monographs:23 - Indexing 5479 monographs into 'cardiometabolic_drugs'
2026-04-22 03:42:58 | INFO     | pharma_intel.rag.embeddings:get_embedder:20 - Loading embedding model: BAAI/bge-m3
2026-04-22 03:42:58 | INFO     | pharma_intel.rag.embeddings:get_embedder:21 - First run will download the model (~2GB for bge-m3). Subsequent runs use cache.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/content/pharma-intelligence/src/pharma_intel/rag/embeddings.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  logger.info(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
2026-04-22 03:43:53 | INFO     | pharma_intel.rag.embeddings:get_embedder:28 - Embedding dim: 1024
2026-04-22 03:43:53 | INFO     | pharma_intel.rag.embeddings:embed_texts:35 - Embedding 5479 texts (batch_size=64)...


Batches:   0%|          | 0/86 [00:00<?, ?it/s]

2026-04-22 03:46:27 | INFO     | pharma_intel.rag.vectorstore:__init__:39 - VectorStore ready — collection='cardiometabolic_drugs', count=0
2026-04-22 03:46:42 | INFO     | pharma_intel.rag.vectorstore:add:72 - Indexed 5479 documents (total in collection: 5479)



✓ Indexing took 224.3 seconds
✓ Collection size: 5479


## 6. Test query on Colab

In [9]:
from pharma_intel.rag import RAGEngine

engine = RAGEngine(top_k=5)
result = engine.answer('What patents cover empagliflozin and when do they expire?')

print('--- ANSWER ---')
print(result.answer)
print(f'\n--- Retrieved {len(result.retrieved)} sources ---')
for c in result.retrieved:
    print(f'  [{c.doc_id}] {c.metadata["ingredient"]} — score={c.score:.3f}')

2026-04-22 03:46:42 | INFO     | pharma_intel.rag.vectorstore:__init__:39 - VectorStore ready — collection='cardiometabolic_drugs', count=5479
2026-04-22 03:46:42 | INFO     | pharma_intel.rag.llm.factory:get_llm:27 - Initializing LLM provider: thaillm
2026-04-22 03:46:42 | WARNING  | pharma_intel.rag.llm.thaillm_provider:__init__:46 - ThaiLLM endpoint uses HTTP (not HTTPS). API key will be transmitted unencrypted. Do not use on untrusted networks.
2026-04-22 03:46:42 | INFO     | pharma_intel.rag.query:__init__:65 - RAGEngine ready — collection='cardiometabolic_drugs', llm=thaillm/openthaigpt-thaillm-8b-instruct-v7.2
2026-04-22 03:46:42 | INFO     | pharma_intel.rag.embeddings:embed_texts:35 - Embedding 1 texts (batch_size=32)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-22 03:46:42 | INFO     | pharma_intel.rag.query:retrieve:81 - Retrieved 5 chunks (top score: 0.638)
2026-04-22 03:46:42 | INFO     | pharma_intel.rag.llm.thaillm_provider:complete:83 - Calling ThaiLLM model=openthaigpt-thaillm-8b-instruct-v7.2


--- ANSWER ---
<think>
คำถามคือ "What patents cover empagliflozin and when do they expire?" หรือ "มีสิทธิบัตรใดบ้างที่ครอบคลุม empagliflozin และสิทธิบัตรเหล่านั้นหมดอายุเมื่อใด" คำตอบต้องมาจากข้อมูลใน context ที่ให้มาเท่านั้น ห้ามสร้างข้อมูลที่ไม่มีใน context

จาก context ที่ให้มา มีข้อมูลเกี่ยวกับสิทธิบัตรของยา TRIJARDY XR และ SYNJARDY XR ซึ่งมี active ingredient คือ empagliflozin และ metformin hydrochloride ดังนั้นจะต้องค้นหาสิทธิบัตรที่เกี่ยวข้องกับ empagliflozin โดยเฉพาะ

ขั้นตอนการตอบคำถาม:
1. ค้นหาสิทธิบัตรที่ระบุว่าครอบคลุม "drug substance" หรือ "drug product" ที่มี empagliflozin เป็นส่วนประกอบ
2. ระบุเลขสิทธิบัตร วันหมดอายุ และแหล่งข้อมูล (doc_id)
3. จัดเรียงข้อมูลเป็นรายการ (bullet points) เพื่อให้อ่านง่าย
4. ตรวจสอบให้แน่ใจว่าไม่มีข้อมูลที่ไม่เกี่ยวข้องหรือสร้างขึ้นเอง
</think>

สิทธิบัตรที่ครอบคลุม empagliflozin และกำหนดวันหมดอายุมีดังนี้:

*   **Patent 7713938**: ครอบคลุม "drug substance" และ "drug product" หมดอายุวันที่ **Apr 15, 2027** [doc_id: OB-212614-001, OB-212614-00

## 7. Run the Experiment 02 bilingual evaluation set

This cell runs a compact bilingual sanity set and writes structured artifacts that you can use to draft the Experiment 02 results section.

Outputs written to `DATA_DIR/processed`:

- `experiment_02_eval_results.csv`: one row per query, including answer text, retrieved sources, latency, token usage, and blank manual review columns
- `experiment_02_eval_results.json`: the same run in JSON form
- `experiment_02_eval_summary.md`: an auto-generated markdown summary with basic operational metrics

The notebook computes the mechanical parts automatically. Manual reviewer columns remain blank so you can score retrieval relevance, answer grounding, and citation quality after the run.

In [10]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd

EVAL_QUERIES = [
    {
        'query_id': 'EN01',
        'language': 'EN',
        'category': 'core',
        'expected_information_type': 'patents and expiry dates',
        'expected_focus': 'empagliflozin',
        'question': 'What patents cover empagliflozin and when do they expire?',
    },
    {
        'query_id': 'EN02',
        'language': 'EN',
        'category': 'core',
        'expected_information_type': 'approved products and applicants',
        'expected_focus': 'SGLT2 inhibitors',
        'question': 'List SGLT2 inhibitors approved by FDA with their manufacturers.',
    },
    {
        'query_id': 'EN03',
        'language': 'EN',
        'category': 'core',
        'expected_information_type': 'generic manufacturers',
        'expected_focus': 'metformin',
        'question': 'Compare generic manufacturers of metformin.',
    },
    {
        'query_id': 'EN04',
        'language': 'EN',
        'category': 'core',
        'expected_information_type': 'exclusivity records',
        'expected_focus': 'dapagliflozin',
        'question': 'What exclusivities are listed for dapagliflozin products?',
    },
    {
        'query_id': 'EN05',
        'language': 'EN',
        'category': 'core',
        'expected_information_type': 'combination products',
        'expected_focus': 'empagliflozin',
        'question': 'Show FDA-approved combination products that include empagliflozin.',
    },
    {
        'query_id': 'TH01',
        'language': 'TH',
        'category': 'core',
        'expected_information_type': 'patents and expiry dates',
        'expected_focus': 'dapagliflozin',
        'question': 'สิทธิบัตรของ dapagliflozin หมดเมื่อไหร่',
    },
    {
        'query_id': 'TH02',
        'language': 'TH',
        'category': 'core',
        'expected_information_type': 'approved products and applicants',
        'expected_focus': 'SGLT2 inhibitors',
        'question': 'มียา SGLT2 inhibitors ตัวใดบ้างที่ได้รับอนุมัติจาก FDA และผู้ผลิตคือใคร',
    },
    {
        'query_id': 'TH03',
        'language': 'TH',
        'category': 'core',
        'expected_information_type': 'generic manufacturers',
        'expected_focus': 'metformin',
        'question': 'เปรียบเทียบผู้ผลิตยาสามัญของ metformin',
    },
    {
        'query_id': 'TH04',
        'language': 'TH',
        'category': 'core',
        'expected_information_type': 'combination products',
        'expected_focus': 'empagliflozin',
        'question': 'มีผลิตภัณฑ์ผสมใดบ้างที่มี empagliflozin',
    },
    {
        'query_id': 'TH05',
        'language': 'TH',
        'category': 'core',
        'expected_information_type': 'dosage form and route',
        'expected_focus': 'atorvastatin',
        'question': 'รูปแบบยาและวิธีใช้ของ atorvastatin ที่พบใน Orange Book คืออะไร',
    },
    {
        'query_id': 'ST01',
        'language': 'EN',
        'category': 'no-answer',
        'expected_information_type': 'abstention behavior',
        'expected_focus': 'out-of-scope Thailand procurement',
        'question': 'What does the Orange Book say about Thai hospital procurement of empagliflozin?',
    },
    {
        'query_id': 'ST02',
        'language': 'TH',
        'category': 'no-answer',
        'expected_information_type': 'abstention behavior',
        'expected_focus': 'out-of-scope Thailand market share',
        'question': 'ส่วนแบ่งตลาดในประเทศไทยปี 2025 ของยา SGLT2 inhibitors แต่ละตัวคืออะไร',
    },
]

output_dir = Path(os.environ['DATA_DIR']) / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
results_csv_path = output_dir / 'experiment_02_eval_results.csv'
results_json_path = output_dir / 'experiment_02_eval_results.json'
summary_md_path = output_dir / 'experiment_02_eval_summary.md'

thai_pattern = re.compile(r'[\u0E00-\u0E7F]')
citation_pattern = re.compile(r'\[([^\]]+)\]')
abstain_markers = [
    'not available',
    'no relevant documents found',
    'not found in the context',
    'does not contain',
    'ไม่พบ',
    'ไม่มีข้อมูล',
    'ไม่อยู่ในข้อมูล',
]

def detect_response_language(text: str) -> str:
    thai_chars = len(thai_pattern.findall(text))
    ascii_letters = sum(1 for char in text if char.isascii() and char.isalpha())
    if thai_chars >= 10 or thai_chars > max(5, ascii_letters * 0.2):
        return 'TH'
    return 'EN'

def extract_citations(answer: str) -> list[str]:
    return list(dict.fromkeys(citation_pattern.findall(answer)))

engine = RAGEngine(top_k=5)
records = []

for item in EVAL_QUERIES:
    print(f"Running {item['query_id']} ({item['language']})...")
    start = time.perf_counter()
    result = engine.answer(item['question'])
    latency_seconds = time.perf_counter() - start

    retrieved_doc_ids = [chunk.doc_id for chunk in result.retrieved]
    retrieved_ingredients = [chunk.metadata.get('ingredient', 'N/A') for chunk in result.retrieved]
    retrieved_scores = [round(chunk.score, 4) for chunk in result.retrieved]
    citations = extract_citations(result.answer)
    response_language = detect_response_language(result.answer)
    answer_lower = result.answer.lower()
    looks_like_abstention = any(marker in answer_lower for marker in abstain_markers)
    citations_supported = bool(citations) and set(citations).issubset(set(retrieved_doc_ids))

    records.append({
        'query_id': item['query_id'],
        'language': item['language'],
        'category': item['category'],
        'expected_information_type': item['expected_information_type'],
        'expected_focus': item['expected_focus'],
        'question': item['question'],
        'provider': result.provider,
        'model': result.model,
        'latency_seconds': round(latency_seconds, 3),
        'response_language': response_language,
        'language_match_heuristic': response_language == item['language'],
        'citation_present': bool(citations),
        'citation_count': len(citations),
        'citations_supported_by_retrieval': citations_supported,
        'looks_like_abstention': looks_like_abstention,
        'retrieved_doc_ids': ' | '.join(retrieved_doc_ids),
        'retrieved_ingredients': ' | '.join(retrieved_ingredients),
        'retrieved_scores': ' | '.join(str(score) for score in retrieved_scores),
        'top_score': retrieved_scores[0] if retrieved_scores else None,
        'input_tokens': result.input_tokens,
        'output_tokens': result.output_tokens,
        'answer': result.answer,
        'manual_retrieval_relevance': '',
        'manual_answer_grounding': '',
        'manual_citation_quality': '',
        'manual_notes': '',
    })

results_df = pd.DataFrame(records)
results_df.to_csv(results_csv_path, index=False, encoding='utf-8-sig')
results_df.to_json(results_json_path, orient='records', indent=2, force_ascii=False)

summary = {
    'total_queries': int(len(results_df)),
    'english_queries': int((results_df['language'] == 'EN').sum()),
    'thai_queries': int((results_df['language'] == 'TH').sum()),
    'mean_latency_seconds': round(float(results_df['latency_seconds'].mean()), 3),
    'p95_latency_seconds': round(float(results_df['latency_seconds'].quantile(0.95)), 3),
    'citation_presence_rate': round(float(results_df['citation_present'].mean()), 3),
    'citation_supported_rate': round(float(results_df['citations_supported_by_retrieval'].mean()), 3),
    'language_match_rate_heuristic': round(float(results_df['language_match_heuristic'].mean()), 3),
    'abstention_rate': round(float(results_df['looks_like_abstention'].mean()), 3),
    'mean_top_score': round(float(results_df['top_score'].dropna().mean()), 3),
}

summary_lines = [
    '# Experiment 02 Evaluation Run',
    '',
    'This summary was generated automatically from the Colab evaluation cell.',
    '',
    '## Run Metrics',
    '',
    f"- Total queries: {summary['total_queries']}",
    f"- English queries: {summary['english_queries']}",
    f"- Thai queries: {summary['thai_queries']}",
    f"- Mean latency (s): {summary['mean_latency_seconds']}",
    f"- P95 latency (s): {summary['p95_latency_seconds']}",
    f"- Citation presence rate: {summary['citation_presence_rate']}",
    f"- Citation supported-by-retrieval rate: {summary['citation_supported_rate']}",
    f"- Heuristic language match rate: {summary['language_match_rate_heuristic']}",
    f"- Abstention rate: {summary['abstention_rate']}",
    f"- Mean top retrieval score: {summary['mean_top_score']}",
    '',
    '## Artifacts',
    '',
    f"- CSV: {results_csv_path}",
    f"- JSON: {results_json_path}",
    f"- Summary: {summary_md_path}",
    '',
    '## Manual Review Reminder',
    '',
    'Fill the manual review columns in the CSV after checking retrieval relevance, grounding, citation quality, and any failure notes.',
]

summary_md_path.write_text('\n'.join(summary_lines), encoding='utf-8')

display_columns = [
    'query_id',
    'language',
    'category',
    'latency_seconds',
    'citation_present',
    'citations_supported_by_retrieval',
    'language_match_heuristic',
    'looks_like_abstention',
    'retrieved_doc_ids',
    'answer',
]

display(results_df[display_columns])
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f'\nSaved CSV to: {results_csv_path}')
print(f'Saved JSON to: {results_json_path}')
print(f'Saved markdown summary to: {summary_md_path}')

2026-04-22 03:47:04 | INFO     | pharma_intel.rag.vectorstore:__init__:39 - VectorStore ready — collection='cardiometabolic_drugs', count=5479
2026-04-22 03:47:04 | INFO     | pharma_intel.rag.llm.factory:get_llm:27 - Initializing LLM provider: thaillm
2026-04-22 03:47:04 | WARNING  | pharma_intel.rag.llm.thaillm_provider:__init__:46 - ThaiLLM endpoint uses HTTP (not HTTPS). API key will be transmitted unencrypted. Do not use on untrusted networks.
2026-04-22 03:47:04 | INFO     | pharma_intel.rag.query:__init__:65 - RAGEngine ready — collection='cardiometabolic_drugs', llm=thaillm/openthaigpt-thaillm-8b-instruct-v7.2
2026-04-22 03:47:04 | INFO     | pharma_intel.rag.embeddings:embed_texts:35 - Embedding 1 texts (batch_size=32)...


Running EN01 (EN)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-22 03:47:04 | INFO     | pharma_intel.rag.query:retrieve:81 - Retrieved 5 chunks (top score: 0.638)
2026-04-22 03:47:04 | INFO     | pharma_intel.rag.llm.thaillm_provider:complete:83 - Calling ThaiLLM model=openthaigpt-thaillm-8b-instruct-v7.2
2026-04-22 03:47:15 | INFO     | pharma_intel.rag.embeddings:embed_texts:35 - Embedding 1 texts (batch_size=32)...


Running EN02 (EN)...


...

Running ST02 (TH)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-22 03:48:10 | INFO     | pharma_intel.rag.query:retrieve:81 - Retrieved 5 chunks (top score: 0.451)
2026-04-22 03:48:10 | INFO     | pharma_intel.rag.llm.thaillm_provider:complete:83 - Calling ThaiLLM model=openthaigpt-thaillm-8b-instruct-v7.2


,query_id,language,category,latency_seconds,citation_present,citations_supported_by_retrieval,language_match_heuristic,looks_like_abstention,retrieved_doc_ids,answer
0,EN01,EN,core,10.936,True,False,False,False,OB-212614-003 | OB-212614-002 | OB-212614-001 ...,"<think>\nคำถามคือ ""What patents cover empaglif..."
1,EN02,EN,core,4.926,False,False,False,True,OB-206079-002 | OB-206079-003 | OB-206079-001 ...,"<think>\nคำถามคือ ""List SGLT2 inhibitors appro..."
2,EN03,EN,core,7.263,True,True,False,False,OB-090888-003 | OB-078170-003 | OB-212969-002 ...,<think>\nคำถามนี้ต้องการให้เปรียบเทียบผู้ผลิตย...
3,EN04,EN,core,3.373,True,False,False,False,OB-202293-002 | OB-202293-001 | OB-211506-002 ...,"<think>\nคำถามคือ ""What exclusivities are list..."
4,EN05,EN,core,5.222,True,True,False,False,OB-212138-002 | OB-212138-001 | OB-078012-001 ...,"<think>\nคำถามคือ ""แสดงผลิตภัณฑ์ที่รวมกันที่ได..."
5,TH01,TH,core,10.832,True,False,True,False,OB-202293-002 | OB-202293-001 | OB-205649-001 ...,"<think>\nคำถามคือ ""สิทธิบัตรของ dapagliflozin ..."
6,TH02,TH,core,3.482,False,False,True,True,OB-205936-001 | OB-218085-001 | OB-086513-001 ...,"<think>\nคำถามคือ ""มียา SGLT2 inhibitors ตัวใด..."
7,TH03,TH,core,6.907,True,True,True,False,OB-212677-001 | OB-076869-001 | OB-090888-003 ...,<think>\nคำถามนี้ต้องการให้เปรียบเทียบผู้ผลิตย...
8,TH04,TH,core,6.108,True,False,True,False,OB-212138-002 | OB-212138-001 | OB-212198-001 ...,"<think>\nคำถามคือ ""มีผลิตภัณฑ์ผสมใดบ้างที่มี e..."
9,TH05,TH,core,3.484,True,True,True,False,OB-078735-004 | OB-078735-002 | OB-078735-005 ...,<think>\nคำถามนี้ต้องการทราบรูปแบบยาและวิธีใช้...


{
  "total_queries": 12,
  "english_queries": 6,
  "thai_queries": 6,
  "mean_latency_seconds": 6.135,
  "p95_latency_seconds": 10.879,
  "citation_presence_rate": 0.75,
  "citation_supported_rate": 0.333,
  "language_match_rate_heuristic": 0.5,
  "abstention_rate": 0.333,
  "mean_top_score": 0.549
}

Saved CSV to: /content/drive/MyDrive/pharma-intelligence/data/processed/experiment_02_eval_results.csv
Saved JSON to: /content/drive/MyDrive/pharma-intelligence/data/processed/experiment_02_eval_results.json
Saved markdown summary to: /content/drive/MyDrive/pharma-intelligence/data/processed/experiment_02_eval_summary.md


## 8. Export ChromaDB as a verified zip

The next cell creates a zip archive from the completed `chroma_db` directory, verifies that the expected Chroma index files are present, and then downloads the archive to your local machine.

Use this export step instead of manually copying the folder from Drive. Manual copies can miss HNSW index files such as `header.bin`, `data_level0.bin`, `length.bin`, and `link_lists.bin`, which makes the local index unreadable.

After the download finishes, extract the archive locally and point `CHROMA_PERSIST_DIR` at the extracted `chroma_db` directory.

In [11]:
import os
import shutil
import zipfile
from pathlib import Path
from google.colab import files

persist_dir = Path(os.environ['CHROMA_PERSIST_DIR'])
archive_base = Path('/content/chroma_db_export')
zip_path = archive_base.with_suffix('.zip')

if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(archive_base), 'zip', root_dir=persist_dir.parent, base_dir=persist_dir.name)

required_files = {
    'chroma.sqlite3',
    'index_metadata.pickle',
    'header.bin',
    'data_level0.bin',
    'length.bin',
    'link_lists.bin',
}

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive_files = {Path(name).name for name in archive.namelist() if not name.endswith('/')}

missing = required_files - archive_files
if missing:
    raise RuntimeError(f'Zip archive is missing required Chroma files: {sorted(missing)}')

print(f'Zip created: {zip_path}')
print(f'Zip size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB')
print('Verified files present in archive:')
for name in sorted(required_files):
    print(f'  - {name}')

files.download(str(zip_path))

Zip created: /content/chroma_db_export.zip
Zip size: 44.7 MB
Verified files present in archive:
  - chroma.sqlite3
  - data_level0.bin
  - header.bin
  - index_metadata.pickle
  - length.bin
  - link_lists.bin


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>